# Data Cleaning & Unification

This notebook processes the raw refugee hosting messages data:
- Loads 3 Excel sheets
- Unifies into single dataset
- Cleans text and standardizes fields
- Engineers features for analysis
- Calculates quality scores
- Saves processed outputs

**Output**: Clean, unified dataset ready for emotion analysis and avatar generation

In [ ]:
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")

## 1. Load Raw Data

In [ ]:
# Find data file
data_dir = Path('../data/raw')
excel_files = list(data_dir.glob('*.xlsx'))

if not excel_files:
    raise FileNotFoundError("No Excel file found in data/raw/")

data_path = excel_files[0]
print(f"📂 Loading: {data_path.name}")

# Load all sheets
familles_1 = pd.read_excel(data_path, sheet_name='Familles 1')
familles_2 = pd.read_excel(data_path, sheet_name='Familles 2')
nouvelle_plateforme = pd.read_excel(data_path, sheet_name='Nouvelle Plateforme')

print(f"\n✓ Familles 1: {len(familles_1):,} rows, {len(familles_1.columns)} columns")
print(f"✓ Familles 2: {len(familles_2):,} rows, {len(familles_2.columns)} columns")
print(f"✓ Nouvelle Plateforme: {len(nouvelle_plateforme):,} rows, {len(nouvelle_plateforme.columns)} columns")
print(f"\n📊 Total messages: {len(familles_1) + len(familles_2) + len(nouvelle_plateforme):,}")

## 2. Standardize Column Names

In [ ]:
# Column mapping for unification
column_mapping = {
    'familles_1': {
        'CP': 'postal_code',
        'Motivations / Pourquoi CALM': 'motivation_text',
        "Type d'espace mis à disposition ": 'space_type',
        'Parlez nous de vous': 'personal_description',
        "Temps d'accueil": 'hosting_duration',
        'avec enfants ? ': 'accepts_children',
        'Partage de réseau/métier ? ': 'shares_network'
    },
    'familles_2': {
        'CP': 'postal_code',
        'Age': 'age',
        'Motivations / Pourquoi CALM': 'motivation_text',
        "Type d'espace mis à disposition ": 'space_type',
        'Taille': 'space_size_m2',
        'Parlez nous de vous': 'personal_description',
        "Temps d'accueil": 'hosting_duration',
        'avec enfants ? ': 'accepts_children',
        'Enseignement métier / partage réseau': 'teaches_or_shares',
        "Capacité d'accueil": 'hosting_capacity',
        'Autres infos': 'other_info'
    },
    'nouvelle_plateforme': {
        'CP': 'postal_code',
        'Date de naissance': 'birth_date',
        'Motivations / Pourquoi CALM': 'motivation_text',
        "Type d'espace mis à disposition ": 'space_type',
        'Taille': 'space_size_m2',
        'Parlez nous de vous': 'personal_description',
        "Temps d'accueil": 'hosting_duration',
        'avec enfants ? ': 'accepts_children',
        'Enseignement métier': 'teaches_job',
        'Partage de réseau': 'shares_network',
        "Capacité d'accueil": 'hosting_capacity',
        'Autres infos': 'other_info'
    }
}

# Rename columns
f1 = familles_1.rename(columns=column_mapping['familles_1'])
f2 = familles_2.rename(columns=column_mapping['familles_2'])
np_df = nouvelle_plateforme.rename(columns=column_mapping['nouvelle_plateforme'])

# Add source identifier
f1['source_sheet'] = 'Familles 1'
f2['source_sheet'] = 'Familles 2'
np_df['source_sheet'] = 'Nouvelle Plateforme'

print("✓ Columns standardized")
print(f"\nFamilles 1 columns: {list(f1.columns)}")
print(f"\nFamilles 2 columns: {list(f2.columns)}")
print(f"\nNouvelle Plateforme columns: {list(np_df.columns)}")

## 3. Unify Datasets

In [ ]:
# Combine all sheets (union of all columns)
df = pd.concat([f1, f2, np_df], ignore_index=True, sort=False)

# Create unique message IDs
def create_message_id(row, idx):
    prefix = {'Familles 1': 'F1', 'Familles 2': 'F2', 'Nouvelle Plateforme': 'NP'}[row['source_sheet']]
    return f"{prefix}_{idx:04d}"

df['message_id'] = [create_message_id(row, idx) for idx, row in df.iterrows()]

# Reorder columns (ID first)
cols = ['message_id', 'source_sheet'] + [col for col in df.columns if col not in ['message_id', 'source_sheet']]
df = df[cols]

print(f"✓ Unified dataset created: {len(df):,} messages")
print(f"✓ Columns: {len(df.columns)}")
print(f"\nColumn list:\n{list(df.columns)}")

## 4. Clean Text Fields

In [ ]:
def clean_text(text):
    """Clean text while preserving authenticity"""
    if pd.isna(text) or not isinstance(text, str):
        return None
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    # Remove empty strings
    if len(text) == 0:
        return None
    
    return text

# Clean text columns
text_columns = ['motivation_text', 'personal_description', 'other_info', 'space_type']

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)
        print(f"✓ Cleaned: {col}")

print(f"\n✓ Text cleaning complete")

## 5. Feature Engineering

In [ ]:
# Extract department from postal code
def extract_department(cp):
    if pd.isna(cp):
        return None
    cp_str = str(int(cp)) if isinstance(cp, float) else str(cp)
    if len(cp_str) >= 2:
        return cp_str[:2]
    return None

df['department'] = df['postal_code'].apply(extract_department)

# Convert postal code to string (preserve leading zeros)
df['postal_code'] = df['postal_code'].apply(lambda x: str(int(x)).zfill(5) if pd.notna(x) else None)

print(f"✓ Extracted department codes")
print(f"  Unique departments: {df['department'].nunique()}")

In [ ]:
# Text length
df['text_length'] = df['motivation_text'].apply(lambda x: len(x) if pd.notna(x) else 0)

# Boolean flags
df['has_motivation'] = df['motivation_text'].notna()
df['has_description'] = df['personal_description'].notna()
df['has_other_info'] = df['other_info'].notna()

print("✓ Added text length and flags")
print(f"  Messages with motivation: {df['has_motivation'].sum():,} ({df['has_motivation'].mean()*100:.1f}%)")
print(f"  Messages with description: {df['has_description'].sum():,} ({df['has_description'].mean()*100:.1f}%)")
print(f"  Average text length: {df['text_length'].mean():.0f} chars")

In [ ]:
# Categorize hosting duration
def categorize_duration(duration):
    if pd.isna(duration):
        return 'unknown'
    
    duration_lower = str(duration).lower()
    
    if 'semaine' in duration_lower and '2' in duration_lower:
        return 'short'  # 2 weeks
    elif '3 mois' in duration_lower or '3-6' in duration_lower:
        return 'medium'  # 3-6 months
    elif '6' in duration_lower or '9' in duration_lower:
        return 'medium-long'  # 6-9 months
    elif 'an' in duration_lower or 'année' in duration_lower:
        return 'long'  # 1+ year
    elif 'autant' in duration_lower or 'limite' in duration_lower or 'nécessaire' in duration_lower:
        return 'unlimited'
    else:
        return 'other'

df['duration_category'] = df['hosting_duration'].apply(categorize_duration)

print("✓ Categorized hosting duration")
print(f"\nDuration distribution:\n{df['duration_category'].value_counts()}")

In [ ]:
# Categorize space type
def categorize_space(space):
    if pd.isna(space):
        return 'unknown'
    
    space_lower = str(space).lower()
    
    if 'chambre' in space_lower:
        return 'room'
    elif 'studio' in space_lower:
        return 'studio'
    elif 'logement' in space_lower or 'appartement' in space_lower:
        return 'autonomous'
    elif 'canapé' in space_lower or 'salon' in space_lower:
        return 'sofa'
    else:
        return 'other'

df['space_category'] = df['space_type'].apply(categorize_space)

print("✓ Categorized space type")
print(f"\nSpace distribution:\n{df['space_category'].value_counts()}")

## 6. Quality Scoring

In [ ]:
def calculate_quality_score(row):
    """Calculate message quality score (0-100)"""
    score = 0
    
    # Essential: Has motivation text (40 points)
    if row['has_motivation']:
        score += 40
        # Bonus for substantial text
        if row['text_length'] > 100:
            score += 5
        if row['text_length'] > 200:
            score += 5
    
    # Important: Has personal description (20 points)
    if row['has_description']:
        score += 20
    
    # Useful metadata (30 points total)
    if pd.notna(row['postal_code']):
        score += 10
    if pd.notna(row.get('age')) or pd.notna(row.get('birth_date')):
        score += 5
    if pd.notna(row.get('space_type')):
        score += 5
    if pd.notna(row['hosting_duration']):
        score += 5
    if pd.notna(row.get('accepts_children')):
        score += 5
    
    return min(score, 100)  # Cap at 100

df['quality_score'] = df.apply(calculate_quality_score, axis=1)

print("✓ Quality scores calculated")
print(f"\nQuality score distribution:")
print(df['quality_score'].describe())
print(f"\nHigh quality (>= 70): {(df['quality_score'] >= 70).sum():,} messages")
print(f"Medium quality (50-69): {((df['quality_score'] >= 50) & (df['quality_score'] < 70)).sum():,} messages")
print(f"Low quality (< 50): {(df['quality_score'] < 50).sum():,} messages")

## 7. Data Summary Statistics

In [ ]:
# Generate statistics
stats = {
    'dataset_info': {
        'total_messages': len(df),
        'source_sheets': df['source_sheet'].value_counts().to_dict(),
        'columns': list(df.columns),
        'generated_date': datetime.now().isoformat()
    },
    'text_content': {
        'messages_with_motivation': int(df['has_motivation'].sum()),
        'messages_with_description': int(df['has_description'].sum()),
        'avg_text_length': float(df['text_length'].mean()),
        'median_text_length': float(df['text_length'].median()),
        'max_text_length': int(df['text_length'].max())
    },
    'geography': {
        'unique_departments': int(df['department'].nunique()),
        'top_10_departments': df['department'].value_counts().head(10).to_dict()
    },
    'hosting': {
        'duration_categories': df['duration_category'].value_counts().to_dict(),
        'space_categories': df['space_category'].value_counts().to_dict(),
        'accepts_children': df['accepts_children'].value_counts().to_dict()
    },
    'quality': {
        'avg_quality_score': float(df['quality_score'].mean()),
        'median_quality_score': float(df['quality_score'].median()),
        'high_quality_count': int((df['quality_score'] >= 70).sum()),
        'medium_quality_count': int(((df['quality_score'] >= 50) & (df['quality_score'] < 70)).sum()),
        'low_quality_count': int((df['quality_score'] < 50).sum())
    }
}

print("✓ Statistics generated")
print(json.dumps(stats, indent=2, ensure_ascii=False))

## 8. Save Outputs

In [ ]:
output_dir = Path('../data/processed')
output_dir.mkdir(exist_ok=True)

# 1. Full dataset (CSV) - gitignored
csv_path = output_dir / 'messages_unified.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f"✓ Saved full dataset: {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)")

# 2. Full dataset (Parquet) - gitignored, compressed
parquet_path = output_dir / 'messages_unified.parquet'
df.to_parquet(parquet_path, index=False)
print(f"✓ Saved Parquet: {parquet_path} ({parquet_path.stat().st_size / 1024:.1f} KB)")

# 3. Sample dataset (100 messages) - committed to git
sample_df = df[df['quality_score'] >= 60].sample(n=min(100, len(df)), random_state=42)
sample_path = output_dir / 'sample_messages_100.csv'
sample_df.to_csv(sample_path, index=False, encoding='utf-8')
print(f"✓ Saved sample (100 messages): {sample_path} ({sample_path.stat().st_size / 1024:.1f} KB)")

# 4. Statistics JSON - committed to git
stats_path = output_dir / 'dataset_statistics.json'
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(f"✓ Saved statistics: {stats_path}")

# 5. Column mapping - committed to git
mapping_path = output_dir / 'column_mapping.json'
with open(mapping_path, 'w', encoding='utf-8') as f:
    json.dump(column_mapping, f, indent=2, ensure_ascii=False)
print(f"✓ Saved column mapping: {mapping_path}")

print(f"\n{'='*60}")
print("✅ DATA CLEANING COMPLETE")
print(f"{'='*60}")
print(f"\nProcessed: {len(df):,} messages")
print(f"Average quality score: {df['quality_score'].mean():.1f}/100")
print(f"\nOutputs saved to: {output_dir}")
print("\nNext: Use 03_random_message_viewer.ipynb to explore the data")

In [ ]:
# Quick preview of cleaned data
print("Preview of cleaned data:\n")
display(df[['message_id', 'source_sheet', 'department', 'text_length', 'duration_category', 
            'space_category', 'quality_score']].head(10))